In [1]:
import pandas as pd

test_df = pd.read_csv('~/data/test_data_with_calender_features.csv')


test_df['ds'] = pd.to_datetime(test_df['ds'])

test_df = test_df[test_df['ds'] <= '2024-04-01 22:00:00']

print("First date in test_df:", test_df['ds'].min())
print("Last date in test_df:", test_df['ds'].max())

First date in test_df: 2023-12-31 23:00:00
Last date in test_df: 2024-04-01 22:00:00


In [2]:

test_df.to_csv('test_data_small_with_calender_features.csv', index=False)

In [ ]:
import numpy as np
import holidays
print("holidays module imported successfully!")
import pandas as pd

# Define the cyclic encoding function
def cyclicEncode(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col] / max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col] / max_val)
    return data


# Add calendar features
def add_calendar_features(df):
    german_holidays = holidays.Germany(years=df['ds'].dt.year.unique())
    
    # Basic date features
    df['day_of_week'] = df['ds'].dt.dayofweek  # 0=Monday, 6=Sunday
    df['month'] = df['ds'].dt.month
    df['hour'] = df['ds'].dt.hour
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Holiday feature
    df['is_holiday'] = df['ds'].apply(lambda x: int(x in german_holidays))
    
    # Add cyclic encoding for month, day_of_week, and hour
    df = cyclicEncode(df, 'month', 12)
    df = cyclicEncode(df, 'day_of_week', 7)
    df = cyclicEncode(df, 'hour', 24)
    
    return df

In [ ]:
# Load data from CSV
train_df = pd.read_csv('train_data_Calender.csv')
test_df = pd.read_csv('test_data_Calender.csv')

# Convert timestamp to datetime format
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])

# Rename columns for NeuralForecast
train_df = train_df.rename(columns={'timestamp': 'ds', 'price': 'y'})
test_df = test_df.rename(columns={'timestamp': 'ds', 'price': 'y'})

# Add unique_id column
train_df['unique_id'] = 'electricity_prices'
test_df['unique_id'] = 'electricity_prices'

# Add features to train and test datasets
train_df = add_calendar_features(train_df)
test_df = add_calendar_features(test_df)

# Check date ranges
print("First date in train_df:", train_df['ds'].min())
print("Last date in train_df:", train_df['ds'].max())
print("First date in test_df:", test_df['ds'].min())
print("Last date in test_df:", test_df['ds'].max())